<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/S10_introduction_to_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Neural Network, Hands-On: a CNN that reads handwritten digits (MNIST)

A companion notebook for today's lecture. We take the **digit-classification** example end to end:
**load data → build a CNN → train it → watch the loss curve fall → look inside the network.**

**▶ Before you start:** turn on the free GPU — **Runtime → Change runtime type → Hardware accelerator → GPU → Save**, then **Runtime → Run all**.

Where this sits in the roadmap:
- **Block 1 — Foundations & Training:** loss, backpropagation, SGD/Adam — we *see* these as the training loop and the loss curve.
- **Block 2 — Architectures:** the **CNN** (convolution, pooling, parameter sharing) is the star here.
- The final cell points to where LSTM / Transformer / LLMs / GNNs pick up.

Nothing to install — it all runs on Colab as-is.

In [ ]:
# --- Setup ----------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)
if device.type == 'cpu':
    print('No GPU detected — it still runs, just slower.')
    print('Enable it via Runtime → Change runtime type → GPU.')

# --- Config: keep it small so a live Run all finishes fast ----------------
FAST_DEMO  = True   # True = small subset (seconds). Set False for full MNIST.
N_TRAIN    = 8000   # used only when FAST_DEMO is True
N_TEST     = 2000
EPOCHS     = 5
BATCH_SIZE = 128
LR         = 1e-3

## 1. The data — MNIST handwritten digits

MNIST is 70,000 grayscale images, each **28×28 pixels**, of a single digit **0–9**. The task: given the pixels, predict the digit.

To keep the live demo fast we use a **small subset** — set `FAST_DEMO = False` above to use all 60,000 training images. The first run **downloads** MNIST into `./data` on this Colab machine (a few seconds).

In [ ]:
# Download MNIST and convert images to tensors in [0, 1]
transform = transforms.ToTensor()
train_full = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_full  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

if FAST_DEMO:
    train_ds = Subset(train_full, range(N_TRAIN))
    test_ds  = Subset(test_full,  range(N_TEST))
else:
    train_ds, test_ds = train_full, test_full

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

print('Training images:', len(train_ds), '| Test images:', len(test_ds))
img0, label0 = train_ds[0]
print('One image tensor shape:', tuple(img0.shape), '(channels, height, width)')

# --- Show a grid of example digits ---
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, img, lab in zip(axes.flat, images, labels):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(int(lab))
    ax.axis('off')
fig.suptitle('A handful of MNIST digits (with their labels)')
plt.tight_layout()
plt.show()

## 2. What does a *convolution* do?

A convolution slides a small grid of weights (a **kernel**) across the image and, at each location, computes a weighted sum of the pixels it covers. Different kernels respond to different local patterns — edges, corners, strokes.

Two ideas make CNNs a great fit for images:
- **Parameter sharing** — the *same* kernel is reused at every position, so the network learns that an edge is an edge no matter where it appears, using very few weights.
- **Pooling** — we then shrink each feature map (e.g. keep the max in every 2×2 block), which keeps the strong responses and adds a little position-invariance.

Below we apply one *hand-made* edge-detecting kernel to a digit — no learning yet, just to see the effect.

In [ ]:
# A fixed 3x3 vertical-edge (Sobel) kernel
sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]]).view(1, 1, 3, 3)

img, lab = test_ds[0]
edge = F.conv2d(img.view(1, 1, 28, 28), sobel_x, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img.squeeze(), cmap='gray')
axes[0].set_title('original (digit ' + str(int(lab)) + ')')
axes[0].axis('off')
axes[1].imshow(edge.squeeze(), cmap='gray')
axes[1].set_title('after a vertical-edge kernel')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Build the CNN

A compact convolutional network:

`Conv(1→16) → ReLU → MaxPool → Conv(16→32) → ReLU → MaxPool → Flatten → Linear(128) → ReLU → Linear(10)`

Each conv layer **learns** its own kernels (instead of us hand-picking them). The two pooling steps shrink 28×28 → 14×14 → 7×7. The final linear layer outputs **10 scores**, one per digit.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print('Trainable parameters:', f'{n_params:,}')

# Sanity check: one forward pass on a real batch
xb, yb = next(iter(train_loader))
out = model(xb.to(device))
print('Batch of', xb.size(0), 'images -> output', tuple(out.shape), '(10 class scores each)')

## 4. Train it — loss, backprop, and the optimizer

Training is a loop. For each **batch** of images we:
1. **Forward** — run the images through the network to get 10 scores each.
2. **Loss** — measure how wrong those scores are with **cross-entropy**.
3. **Backward (backpropagation)** — the chain rule gives the gradient of the loss for every weight.
4. **Step** — the optimizer (**Adam**) nudges each weight a little to lower the loss.

We repeat for a few **epochs** (full passes over the data) and record loss and accuracy each time — exactly what the curves below plot.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def run_epoch(net, loader, optimizer=None):
    train = optimizer is not None
    net.train() if train else net.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = net(xb)
        loss = criterion(logits, yb)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        n          += xb.size(0)
    return total_loss / n, correct / n

history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
    te_loss, te_acc = run_epoch(model, test_loader)
    history['train_loss'].append(tr_loss)
    history['test_loss'].append(te_loss)
    history['train_acc'].append(tr_acc)
    history['test_acc'].append(te_acc)
    print('epoch', epoch, '/', EPOCHS,
          '| train loss', round(tr_loss, 3), 'acc', round(tr_acc, 3),
          '| test loss', round(te_loss, 3), 'acc', round(te_acc, 3))

final_acc = history['test_acc'][-1] * 100
print('Final test accuracy:', round(final_acc, 1), '%')

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epochs_range, history['train_loss'], 'o-', label='train')
ax1.plot(epochs_range, history['test_loss'],  'o-', label='test')
ax1.set_xlabel('epoch')
ax1.set_ylabel('cross-entropy loss')
ax1.set_title('Loss curve — down means learning')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'o-', label='train')
ax2.plot(epochs_range, history['test_acc'],  'o-', label='test')
ax2.set_xlabel('epoch')
ax2.set_ylabel('accuracy')
ax2.set_title('Accuracy curve')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Look inside the trained network

Two quick views:
- **Learned filters** — the 16 kernels of the first conv layer. After training, these have become edge / stroke detectors the network discovered on its own.
- **Feature maps** — what those filters produce when they look at one digit.

In [ ]:
# --- The 16 learned first-layer filters ---
w = model.features[0].weight.detach().cpu()   # shape [16, 1, 3, 3]
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, k in zip(axes.flat, w):
    ax.imshow(k.squeeze(), cmap='gray')
    ax.axis('off')
fig.suptitle('First-layer filters the CNN learned')
plt.tight_layout()
plt.show()

# --- Feature maps for one digit ---
img, lab = test_ds[1]
with torch.no_grad():
    fmaps = F.relu(model.features[0](img.view(1, 1, 28, 28).to(device)))
fmaps = fmaps.cpu().squeeze(0)   # shape [16, 28, 28]
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, fm in zip(axes.flat, fmaps):
    ax.imshow(fm, cmap='gray')
    ax.axis('off')
fig.suptitle('Feature maps after conv-1 for the digit ' + str(int(lab)))
plt.tight_layout()
plt.show()

In [ ]:
# Predictions over the whole test set
all_preds, all_true = [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_true.append(yb)
all_preds = torch.cat(all_preds)
all_true  = torch.cat(all_true)

# --- Confusion matrix ---
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(all_true, all_preds):
    cm[int(t), int(p)] += 1

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('predicted')
ax.set_ylabel('true label')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_title('Confusion matrix (test set)')
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', color=color, fontsize=8)
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

# --- A few of the mistakes ---
wrong = (all_preds != all_true).nonzero(as_tuple=True)[0][:8]
if len(wrong) == 0:
    print('No mistakes in this sample — nice!')
else:
    fig, axes = plt.subplots(1, len(wrong), figsize=(1.6 * len(wrong), 2))
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, wrong):
        img, true = test_ds[int(idx)]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title('pred ' + str(int(all_preds[idx])) + ' / true ' + str(int(true)), fontsize=9)
        ax.axis('off')
    fig.suptitle('Where the CNN slipped up')
    plt.tight_layout()
    plt.show()

## 6. (Optional) Why a CNN? Compare against a plain MLP

Block 1 of the lecture built the **MLP** — fully-connected layers that flatten the image into 784 independent pixels, ignoring its 2-D layout. Let us train one the same way and compare. The CNN usually wins, because convolution + parameter sharing are a better match for images.

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128), nn.ReLU(),
            nn.Linear(128, 64),      nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

mlp = SmallMLP().to(device)
mlp_optimizer = torch.optim.Adam(mlp.parameters(), lr=LR)

mlp_test_loss = []
for epoch in range(1, EPOCHS + 1):
    run_epoch(mlp, train_loader, mlp_optimizer)
    te_loss, te_acc = run_epoch(mlp, test_loader)
    mlp_test_loss.append(te_loss)

mlp_acc = te_acc * 100
cnn_acc = history['test_acc'][-1] * 100
print('MLP test accuracy:', round(mlp_acc, 1), '%  |  CNN test accuracy:', round(cnn_acc, 1), '%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs_range, mlp_test_loss,        'o-', label='MLP')
ax1.plot(epochs_range, history['test_loss'], 'o-', label='CNN')
ax1.set_xlabel('epoch')
ax1.set_ylabel('test loss')
ax1.set_title('Test-loss curves')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.bar(['MLP', 'CNN'], [mlp_acc, cnn_acc], color=['#9aa7b5', '#3b6fb0'])
ax2.set_ylabel('test accuracy (%)')
ax2.set_title('Final accuracy')
ax2.set_ylim(min(mlp_acc, cnn_acc) - 5, 100)
for i, v in enumerate([mlp_acc, cnn_acc]):
    ax2.text(i, v + 0.2, str(round(v, 1)), ha='center')
plt.tight_layout()
plt.show()

## 7. Where the lecture goes from here

You have now seen the full deep-learning loop on a real problem: **data → architecture → loss → backprop → optimizer → curves → inspection.**



**Try it yourself:**
1. Set `FAST_DEMO = False` and re-run — how much does full MNIST help?
2. Swap Adam for `torch.optim.SGD(model.parameters(), lr=0.1)` — how does the loss curve change?
3. Add a third conv layer, or more filters — does test accuracy go up?
4. Push `EPOCHS` higher — watch the gap between train and test (overfitting).